# Análisis Exploratorio de Datos (EDA)

In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.width', 1000)
pd.set_option("display.max_columns", None)
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# 1. Análisis general del dataset

## 1.1. Carga de datos

In [ ]:
df_train = pd.read_csv('../data/time-based-splits/train_before_eda.csv')
df_train.head()
df_train.tail()

## 1.2. Tipos de datos estructurados

### Clasificación de Variables
  
  | Variable | Significado | Tipo analítico |
  |---|---|---|
  | `hotel` | Tipo de hotel (City Hotel o Resort Hotel) | Categórica **nominal** (dicotómica)|
  | `is_canceled` | **Objetivo**: 1 = cancelada, 0 = no cancelada | Categórica **binaria** |
  | `lead_time` | Días transcurridos entre la reserva y la llegada | Numérica **discreta** |
  | `arrival_date_year` | Año de la fecha de llegada | Numérica **discreta** / Categórica **ordinal** |
  | `arrival_date_month` | Mes de la fecha de llegada | Categórica **ordinal** |
  | `arrival_date_week_number` | Número de semana del año de llegada | Numérica **discreta** |
  | `arrival_date_day_of_month` | Día del mes de la fecha de llegada | Numérica **discreta** |
  | `stays_in_weekend_nights` | Número de noches de fin de semana (sábado y/o domingo) que el cliente reservó para pernoctar. | Numérica **discreta** (conteo) |
  | `stays_in_week_nights` | Número de noches de semana (lunes a viernes) reservadas.| Numérica **discreta** (conteo) |
  | `adults` | Número de adultos | Numérica **discreta** (conteo) |
  | `children` | Número de niños | Numérica **discreta** (conteo) |
  | `babies` | Número de bebés | Numérica **discreta** (conteo) |
  | `meal` | Tipo de comida reservada  [**SC / Undefined**: Self Catering (sin comida incluida), **BB**: Bed & Breakfast (desayuno),**HB**: Half Board (media pensión: desayuno y otra comida),**FB**: Full Board (pensión completa: desayuno, almuerzo y cena)]| Categórica **ordinal** |
  | `country` | País de origen del cliente en formato ISO de 3 letras (ej., PRT, GBR, ESP). | Categórica **nominal** |
  | `market_segment` | Segmento de mercado del que procede la reserva (ej., Online TA (Agencia Online), Offline TA/TO (Tour Operador clásico), Corporate (Tarifa de empresa), Groups (Reserva en bloque/grupo), Direct (Reserva individual), Complementary, Aviation). ¿Quién o bajo qué estrategia compra el cliente?| Categórica **nominal** |
  | `distribution_channel` | Canal de reserva o distribución utilizado (ej., TA/TO (Agentes de Viaje / Tour Operadores), Direct (Venta directa por web/teléfono del hotel), Corporate (Canal corporativo interno), GDS (Global Distribution System como Amadeus o Sabre)). ¿A través de qué medio técnico/intermediario llegó la reserva al sistema del hotel? | Categórica **nominal** |
  | `is_repeated_guest` | Huésped repetitivo (1 = sí, 0 = no) | Categórica **binaria** |
  | `previous_cancellations` | Reservas previas canceladas | Numérica **discreta** (conteo) |
  | `previous_bookings_not_canceled` | Reservas previas no canceladas | Numérica **discreta** (conteo) |
  | `reserved_room_type` | Tipo de habitación reservada (ej., A, B, C, D, etc.).| Categórica **nominal** |
  | `assigned_room_type` | Tipo de habitación asignada (puede diferir de la reservada por ascensos de categoría o sobreventa)| Categórica **nominal** |
  | `booking_changes` | Cambios realizados en la reserva | Numérica **discreta** (conteo) |
  | `deposit_type` | Garantía económica asociada a la reserva (No Deposit, Non Refund [no reembolsable], Refundable).| Categórica **nominal** |
  | `agent` | ID de la agencia de viajes que tramitó la reserva (Se almacena numéricamente pero su valor es un identificador nomimal). | Texto / identificador |
  | `company` | ID de la empresa/corporación que realizó o pagó la reserva. (Identificador nominal). | Texto / identificador |
  | `days_in_waiting_list` | Días que la reserva estuvo retenida en lista de espera antes de ser confirmada. | Numérica **discreta** |
  | `customer_type` | Tipología de la estancia y del cliente [**Transient**: Reserva individual estándar no asociada a grupo o contrato, **Contract**: Asociada a un contrato previo o tarifa negociada, **Transient-Party**: Reserva individual vinculada a otra reserva dentro de un grupo implícito, **Group**: Reserva formal de grupo]| Categórica **nominal** |
  | `adr` | Tarifa media diaria (ADR: Average Daily Rate) | Numérica **continua** |
  | `required_car_parking_spaces` | Número de cocheras/estacionamientos solicitados por el cliente. | Numérica **discreta** (conteo) |
  | `total_of_special_requests` | Peticiones especiales | Numérica **discreta** (conteo) |
  | `reservation_status` | Estado final de la reserva (Check-Out, Canceled, No-Show).| Categórica **nominal** |
  | `reservation_status_date` | Fecha del estado final | Fecha / Tiempo |
  | `arrival_month_num` | Mes de la fecha de llegada en números | Categórica **ordinal**  |
  | `arrival_date` | Fecha de llegada | Fecha / Tiempo |
  | `booking_date` | Fecha de la reserva | Fecha / Tiempo |

### Variable target

Nuestra variable objetivo es `is_canceled`, donde:
* `1` = La reserva fue cancelada.
* `0` = La reserva no fue cancelada (se concretó o está activa).
El objetivo de negocio es predecir si una reserva será cancelada para poder tomar medidas preventivas.

## 1.3. Estructura y tipos según pandas

In [ ]:
print(f"Dimensiones del dataset de entrenamiento: {df_train.shape}")   # (registros, variables)
df_train.info(memory_usage="deep")
resumen = pd.DataFrame({
    "tipo": df_train.dtypes,
    "n_unicos": df_train.nunique(),
    "n_faltantes": df_train.isna().sum(),
    "%_faltantes": (df_train.isna().mean() * 100).round(2),
  })
resumen

## 1.4. Evaluar missing y calidad
Analizamos las variables con valores nulos (faltantes) y chequeamos posibles inconsistencias (ej. reservas sin huéspedes).

In [ ]:
# Variables con faltantes
nulos = pd.DataFrame({
    "n_faltantes": df_train.isna().sum(),
    "%_faltantes": (df_train.isna().mean() * 100).round(2).sort_values(ascending=False),
  })
print("Variables con faltantes:")
print(nulos[nulos['%_faltantes'] > 0])

# Inconsistencias: reservas sin huéspedes
sin_huespedes = df_train[(df_train['adults'] == 0) & (df_train['children'] == 0) & (df_train['babies'] == 0)]
print(f"\nReservas sin huéspedes: {len(sin_huespedes)}")

# Inconsistencias: ADR negativo
adr_negativo = df_train[df_train['adr'] < 0]
print(f"Reservas con ADR negativo: {len(adr_negativo)}")

In [ ]:
df_train[df_train['adr'] > 400]

In [ ]:
df_train[df_train['adr'] < 0]

In [ ]:
df_train[(df_train['adr'] <= 0) & (df_train['customer_type'] == "Transient")]

In [ ]:
df_train[(df_train['adr'] <= 0) & (df_train['customer_type'] != "Transient")]

In [ ]:
df_train[(df_train['adults'] == 0) & (df_train['children'] == 0) & (df_train['babies'] == 0)]

In [ ]:
df_train[(df_train['days_in_waiting_list'] > 0) ]

In [ ]:
df_train[(df_train['days_in_waiting_list'] > df_train['lead_time']) ]

In [ ]:
df_train[df_train['adults'] == 55]

In [ ]:
df_train[df_train['babies'] > 3]

In [ ]:
df_train[(df_train['children'] > 2) & (df_train['customer_type'] != "Transient-Party") & (df_train['adults'] == 0) ]

In [ ]:
# 1. Calcular total de noches
df_train["total_stay"] = df_train["stays_in_weekend_nights"] + df_train["stays_in_week_nights"]

# 32. Calcular el límite máximo teórico de noches de fin de semana según el total de días
df_train["max_weekend_teorico"] = 2 * (df_train["total_stay"] // 7) + np.minimum(
    df_train["total_stay"] % 7, 2
)

# 3. Identificar incongruencias calendáricas (donde las noches registradas superan el máximo posible)
incongruencias = df_train[df_train["stays_in_weekend_nights"] > df_train["max_weekend_teorico"]]

print(f"Número de incongruencias calendáricas encontradas: {len(incongruencias)}")

sin_noches = df_train[df_train["total_stay"] == 0]
print(f"Registros con 0 noches totales: {len(sin_noches)}")

## 1.5. Detección de outliers

In [ ]:
num_vars = ['adr', 'lead_time', 'adults', 'children', 'babies', 'stays_in_week_nights', 'stays_in_weekend_nights']
outliers_summary = []

for col in num_vars:
    q1 = df_train[col].quantile(0.25)
    q2 = df_train[col].quantile(0.50)
    q3 = df_train[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    n_outliers = df_train[(df_train[col] < lower_bound) | (df_train[col] > upper_bound)].shape[0]
    pct_outliers = (n_outliers / len(df_train)) * 100
    
    outliers_summary.append({
        'Variable': col,
        'Q1': round(q1, 2),
        'Q2': round(q2, 2),
        'Q3': round(q3, 2),
        'IQR': round(iqr, 2),
        'Límite Sup (Q3 + 1.5 IQR)': round(upper_bound, 2),
        'Max': round(df_train[col].max(), 2),
        'Cant. Outliers': n_outliers,
        '% Outliers': round(pct_outliers, 2)
    })

df_outliers = pd.DataFrame(outliers_summary)
print("=== RESUMEN DE OUTLIERS POR IQR ===")
print(df_outliers.to_string(index=False))

# Inspección de outliers extremos específicos (Casos severos)
adr_extremo = df_train[df_train['adr'] > 500]
adultos_extremo = df_train[df_train['adults'] > 10]

print(f"\nReservas con ADR > 1,000 EUR (Errores severos): {len(adr_extremo)}")
print(f"Reservas con más de 10 adultos: {len(adultos_extremo)}")

# 2. Análisis univariado
## 2.1. Resumen estadístico de las variables numéricas y categóricas.

1.` display(df_train.describe())` Por defecto, el método .describe() procesa únicamente las columnas numéricas (int64, float64).Genera una tabla con 8 métricas estadísticas descriptivas para cada columna:

- `count`: Cantidad de valores no nulos. Útil para identificar qué variables numéricas tienen datos faltantes (por ejemplo, agent o company).
- `mean`: Promedio o media aritmética de la columna.
- `std`: Desviación estándar. Mide la dispersión o variabilidad de los datos respecto a la media.
- min: Valor mínimo registrado. Permite detectar anomalías o errores de carga (por ejemplo, valores negativos en la tarifa diaria adr).
- `25%` (Primer cuartil $Q_1$): El 25% de los datos es menor o igual a este valor.
- `50%` (Mediana o Segundo cuartil $Q_2$): El valor central que divide la distribución al 50%.
- `75%` (Tercer cuartil $Q_3$): El 75% de los datos es menor o igual a este valor.
- `max`: Valor máximo registrado. Crucial para detectar atípicos o extremos (por ejemplo, un adr de $5400$ o adults de $55$).

2. `display(df_train.describe(include=['object', 'category']))` Al pasarle el argumento include=['object', 'category'], obligamos a .describe() a evaluar las variables cualitativas o categóricas (cadenas de texto o categorías explícitas de pandas).Genera un resumen específico para el comportamiento cualitativo:

- `count`: Cantidad de registros no nulos. Permite detectar faltantes en variables de texto como country.
- `unique`: Número de categorías o valores distintos (cardinalidad de la variable). Útil para decidir transformaciones (por ejemplo, hotel tiene 2 categorías únicas, mientras que country tiene 169).
- `top`: La categoría que aparece con mayor frecuencia (la moda).
- `freq`: La frecuencia absoluta con la que aparece la categoría top.

In [ ]:
# Resumen numérico
display(df_train.describe())

# Resumen categórico
display(df_train.describe(include=['object', 'category']))

## 2.2. Análisis univariado de la variable target: `is_canceled`
Veamos la distribución de nuestra variable objetivo para entender si hay desbalance de clases.

In [ ]:
print("=== ANÁLISIS UNIVARIADO DE LA VARIABLE TARGET (is_canceled) ===\n")

# 1. Asegurar formato datetime en la fecha de reserva
df_temp = df_train.copy()
df_temp['booking_date'] = pd.to_datetime(df_temp['booking_date'])

# 2. Conteo y Proporciones
counts = df_temp['is_canceled'].value_counts()
percentages = df_temp['is_canceled'].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({
    'Cantidad': counts,
    'Porcentaje (%)': percentages.round(2)
})
summary_df.index = summary_df.index.map({0: 'No Cancelada (0)', 1: 'Cancelada (1)'})
print(summary_df)
print("\n-----------------------------------------------------------")

# 3. Ratio de Desbalance
imbalance_ratio = counts[0] / counts[1]
print(f"Ratio de Desbalance (No Canceladas / Canceladas): {imbalance_ratio:.2f} : 1")

# 4. Visualizaciones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Barras con conteo y % (Sin warnings en Seaborn/Matplotlib)
palette = ['#2b5c8f', '#d95f02']
sns.barplot(
    x=counts.index, 
    y=counts.values, 
    hue=counts.index, 
    palette=palette, 
    legend=False, 
    ax=axes[0]
)
axes[0].set_title('Distribución de Frecuencia del Target', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Estado de Reserva', fontsize=10)
axes[0].set_ylabel('Número de Reservas', fontsize=10)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['No Cancelada (0)', 'Cancelada (1)'])

for p in axes[0].patches:
    h = p.get_height()
    axes[0].annotate(
        f'{int(h):,}\n({h/len(df_temp):.1%})',
        (p.get_x() + p.get_width() / 2., h / 2),
        ha='center', va='center', color='white', fontweight='bold'
    )
                      
# Gráfico 2: Evolución temporal por booking_date (Corrección del accesor .dt)
df_temp['year_month'] = df_temp['booking_date'].dt.to_period('M')
rate_monthly = df_temp.groupby('year_month')['is_canceled'].mean() * 100

rate_monthly.plot(kind='line', ax=axes[1], marker='o', color='#d95f02', linewidth=2)
axes[1].set_title('Evolución Mensual de la Tasa de Cancelación (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Fecha de Reserva (Mes/Año)', fontsize=10)
axes[1].set_ylabel('Tasa de Cancelación (%)', fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

### Ratio de Desbalance (Imbalance Ratio)

Es una métrica simple que expresa cuántas instancias de la clase mayoritaria (en este caso, reservas no canceladas $0$) existen por cada instancia de la clase minoritaria (reservas canceladas $1$).

$$\text{Ratio de Desbalance} = \frac{\text{Cantidad de clase 0 (No Canceladas)}}{\text{Cantidad de clase 1 (Canceladas)}} = \frac{53,830}{20,456} \approx 2.63$$

#### ¿Cómo interpretar este $2.63 : 1$ en la práctica?
1. Interpretación directa: Significa que por cada reserva que se cancela en el dataset de entrenamiento, el hotel registra aproximadamente 2.63 reservas efectivas.
2. Severidad del desbalance:
  - Balanceado (1:1 a 1.5:1): Las clases están representadas casi por igual.
  - Desbalance Moderado (2:1 a 9:1): Aquí se ubica nuestro proyecto (2.63:1). Hay una clase predominante, pero la clase minoritaria sigue teniendo suficiente representación matemática ($27.5\%$).
  - Desbalance Severo (10:1 a 1000:1 o más): Típico en detección de fraudes con tarjetas de crédito o diagnóstico de enfermedades raras ($0.1\%$ de positivos).

#### Implicaciones para las Fases de Scikit-Learn
Haber calculado este ratio de $2.63 : 1$ nos dicta tres decisiones clave para la construcción de nuestros pipelines y modelos:
1. No necesitamos Over-sampling / Under-sampling agresivo: No hace falta usar técnicas complejas como SMOTE o Random Under Sampling, ya que contamos con más de 20,000 ejemplos de la clase de interés ($1$).
2. Ajuste de pesos en los algoritmos (class_weight='balanced'): En modelos como RandomForestClassifier o LogisticRegression, pasaremos el parámetro class_weight='balanced'. Esto le dice al algoritmo que le dé un peso ligeramente mayor (aproximadamente $2.63$ veces más) a las penalizaciones cuando se equivoca prediciendo una cancelación.
3. Selección de Métricas de Evaluación: Si un modelo predijera siempre "No Cancelado" (cero inteligente), obtendría un $72.5\%$ de Accuracy (exactitud), lo cual sería una métrica engañosa. Por ello, evaluaremos con ROC-AUC, Precision, Recall y el F1-Score de la clase $1$.

### Evolución Mensual de la Tasa de Cancelación (%) según la fecha de reserva (`booking_date`)

1. Distorsión inicial por volumen de muestra (Años 2013-2014)
- Comportamiento: Se observan picos pronunciados y volatilidad extrema entre 2013 y finales de 2014 (alcanzando un pico superior al 80% en octubre de 2014).
- Causa de Negocio: Durante este período hay un número insignificante de reservas generadas (apenas entre 1 y 193 reservas por mes, comparado con las 3,000–6,000 mensuales de 2016 y 2017).
- Impacto en ML: Es un sesgo común en datos históricos antiguos. En la etapa de preprocesamiento evaluaremos si filtrar o penalizar estas pocas filas ruidosas pre-2015 para evitar distorsionar los patrones de los modelos.

2. Estabilización de la tasa en el rango 25% – 33% (Periodo Estable: 2016–2017)
- Comportamiento: A partir de enero de 2016, la tasa de cancelación mensual de las reservas se estabiliza de manera notable dentro de una franja de entre el 26% y el 33%.
- Diagnóstico: Indica que el negocio opera bajo un comportamiento estocástico predecible y que no hubo cambios drásticos en las políticas del hotel ni eventos macroeconómicos disruptivos en esos años.

3. Ausencia de Data Leakage en la División Temporal
- Comportamiento: En el tramo final del conjunto de entrenamiento (principios de 2017), la tasa se sitúa en torno al 27.6% - 29.8%, un valor casi idéntico al promedio general de entrenamiento (27.5%) y al del conjunto de test (27.2%).
- Diagnóstico: Confirma que la partición temporal que realizamos es sólida: el modelo no sufrirá concept drift (cambio de concepto o distribución) severo cuando pase a evaluar el conjunto de prueba (test.csv).

## 2.3. Análisis univariado de variables numéricas predictoras

1. `lead_time` (Días de anticipación de la reserva)
- Forma de la distribución: Presenta una asimetría positiva severa (right-skewed). La mayoría de las reservas se realizan con poca anticipación (mediana = 56 días), pero existe una "cola larga" que se extiende hasta 737 días (más de 2 años de anticipación).
- Impacto en ML: Modelos lineales (como Regresión Logística) se benefician de transformaciones logarítmicas o escalados robustos (RobustScaler) en esta variable para atenuar la asimetría.

2. `adr` (Average Daily Rate / Tarifa diaria promedio)Comportamiento general: 
- Presenta una distribución aproximadamente unimodal concentrada entre 68.5 y 126 EUR (mediana = 93.6 EUR).
- Detección de Atípicos (Outliers): 
  - Mínimos anómalos: Existen valores negativos (mínimo = -6.38 EUR) y valores en $0.00$ EUR (reservas complementarias o errores).
  - Máximo atípico severo: Registra un valor extremo de 5,400 EUR (mientras el percentil 99 está en 242 EUR).
- Decisión de Preparación de Datos: En la etapa de limpieza se deberá aplicar un filtro de límites lógicos (por ejemplo, reemplazar o eliminar tarifas $< 0$ o $> 1000$ EUR) o aplicar un escalador insensible a atípicos.

3. Noches de Estadía (`stays_in_week_nights` y `stays_in_weekend_nights`)
- Comportamiento: La aplastante mayoría de los huéspedes se aloga entre 1 y 4 noches totales.
- Atípicos: Existen valores extremos como 50 noches de semana y 19 noches de fin de semana.
- Ingeniería de Características (Feature Engineering): Conviene crear la variable consolidada total_nights = stays_in_week_nights + stays_in_weekend_nights.

4. Ocupantes (`adults`, `children`, `babies`)
- adults: La mediana es de 2 adultos (modalidad de pareja). Sin embargo, hay registros anómalos con hasta 55 adultos en una sola reserva o reservas con 0 adultos.
- children y babies: Distribuciones altamente dispersas compuestas principalmente por ceros. children contiene 4 valores nulos en el dataset original que imputaremos con 0.

In [ ]:
# Definir figura con 3 filas de subplots (Fila 1: Histogramas, Fila 2: Boxplots, Fila 3: Noches y Adultos)
fig, axes = plt.subplots(
    3, 2, 
    figsize=(14, 12), 
    gridspec_kw={'height_ratios': [2, 1, 2]}
)

# -------------------------------------------------------------------------
# 1. LEAD TIME: Histograma + Boxplot
# -------------------------------------------------------------------------
# Histograma
sns.histplot(df_train['lead_time'], kde=True, ax=axes[0, 0], color='#2b5c8f', bins=40)
axes[0, 0].set_title('Distribución de Lead Time (Días de Anticipación)', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('')
axes[0, 0].set_ylabel('Frecuencia')

# Boxplot debajo
sns.boxplot(x=df_train['lead_time'], ax=axes[1, 0], color='#2b5c8f', flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.5})
axes[1, 0].set_xlabel('Días de Anticipación (lead_time)')

# -------------------------------------------------------------------------
# 2. ADR: Histograma + Boxplot (Filtrado < 400 para escala visual)
# -------------------------------------------------------------------------
# Histograma
sns.histplot(df_train[df_train['adr'] < 400]['adr'], kde=True, ax=axes[0, 1], color='#d95f02', bins=40)
axes[0, 1].set_title('Distribución de ADR (Tarifa Diaria < 400 EUR)', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('Frecuencia')

# Boxplot debajo
sns.boxplot(x=df_train[df_train['adr'] < 400]['adr'], ax=axes[1, 1], color='#d95f02', flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.5})
axes[1, 1].set_xlabel('Precio (EUR)')

# -------------------------------------------------------------------------
# 3. NOCHES TOTALES (Histplot discreto)
# -------------------------------------------------------------------------
total_nights = df_train['stays_in_week_nights'] + df_train['stays_in_weekend_nights']
sns.histplot(total_nights[total_nights <= 14], discrete=True, ax=axes[2, 0], color='#2ca02c')
axes[2, 0].set_title('Distribución de Noches Totales (<= 14 noches)', fontsize=11, fontweight='bold')
axes[2, 0].set_xlabel('Noches')
axes[2, 0].set_ylabel('Frecuencia')

# -------------------------------------------------------------------------
# 4. ADULTOS (Countplot)
# -------------------------------------------------------------------------
sns.countplot(x='adults', data=df_train[df_train['adults'] <= 5], ax=axes[2, 1], color='#82589B')
axes[2, 1].set_title('Distribución de Cantidad de Adultos (<= 5)', fontsize=11, fontweight='bold')
axes[2, 1].set_xlabel('Número de Adultos')
axes[2, 1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()


## 2.4. Análisis univariado de variables categóricas predictoras

1.  `hotel`:
- **City Hotel** representa el 60.7% de las reservas (45,065 registros).
- **Resort Hotel** representa el 39.3% (29,221 registros).
- Diagnóstico: Es una variable binaria balanceada que será ideal para One-Hot Encoding (drop='first').

2. `deposit_type`:
- **No Deposit** concentra el 98.5% de los datos (73,184 registros).
- **Non Refund** representa únicamente el 1.4% (1,007 registros) y **Refundable** un 0.1% (95 registros).
- Diagnóstico: Presenta una varianza extremadamente baja. Evaluaremos en el análisis bivariado si la categoría Non Refund tiene un impacto desproporcionado en la tasa de cancelación.

3. `customer_type`:
- Predominan los clientes **Transient** con el 81.0% (60,198 registros), seguidos por **Transient-Party** (14.4%).
- Las categorías **Contract** (4.0%) y **Group** (0.6%) son minoritarias.

4. `market_segment`:
- El canal predominante es **Online TA** (Agencias de Viaje Online, como Booking o Expedia) con más del 58% de las reservas, seguido por **Offline TA/TO** y reservas directas (**Direct**).
- Las categorías **Aviation** y **Undefined** tienen frecuencias casi nulas.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. hotel (Baja cardinalidad)
sns.countplot(data=df_train, x='hotel', ax=axes[0, 0], palette=['#41b6c4', '#2b5c8f'], hue='hotel')
axes[0, 0].set_title('Distribución por Hotel', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Hotel')
axes[0, 0].set_ylabel('Cantidad de Reservas')
for p in axes[0, 0].patches:
    h = p.get_height()
    axes[0, 0].annotate(f'{int(h):,}\n({h/len(df_train):.1%})',
                        (p.get_x() + p.get_width() / 2., h / 2),
                        ha='center', va='center', color='white', fontweight='bold')

# 2. deposit_type (Baja cardinalidad - Altamente sesgada)
sns.countplot(data=df_train, x='deposit_type', ax=axes[0, 1], color='#d95f02')
axes[0, 1].set_title('Distribución por Deposit Type', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Tipo de Depósito')
axes[0, 1].set_ylabel('Cantidad de Reservas')

# 3. customer_type (Baja cardinalidad)
sns.countplot(data=df_train, x='customer_type', ax=axes[1, 0], color='#2ca02c')
axes[1, 0].set_title('Distribución por Customer Type', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Tipo de Cliente')
axes[1, 0].set_ylabel('Cantidad de Reservas')

# 4. market_segment (Media cardinalidad)
order_ms = df_train['market_segment'].value_counts().index
sns.countplot(data=df_train, y='market_segment', order=order_ms, ax=axes[1, 1], color='#82589B')
axes[1, 1].set_title('Distribución por Market Segment', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Cantidad de Reservas')
axes[1, 1].set_ylabel('Segmento de Mercado')

plt.tight_layout()
plt.show()

# 3. Análisis bivariado

## 3.1. Numérica × numérica: correlación

Al calcular la matriz de correlación sobre las variables cuantitativas del conjunto de entrenamiento, destacan los siguientes coeficientes respecto a `is_canceled`:
1. Las variables numéricas con mayor correlación **POSITIVA** con la cancelación:
- lead_time ($+0.19$): Cuanto mayor es el tiempo de anticipación, mayor es la tendencia a cancelar.
- adr ($+0.12$): A mayor tarifa diaria promedio, mayor es el riesgo de cancelación.
- stays_in_week_nights ($+0.08$) y adults ($+0.07$): Estadías más largas y grupos de adultos más numerosos incrementan ligeramente el riesgo.
2. Las variables numéricas con mayor correlación **NEGATIVA** con la cancelación:
- required_car_parking_spaces ($-0.18$): Pedir espacio de estacionamiento reduce drásticamente la probabilidad de cancelación (el cliente que viaja en auto y reserva parking tiene altísima intención de asistir).
- total_of_special_requests ($-0.12$): Cuantas más peticiones especiales realiza el cliente (ej. cuna, cama alta), menor es la probabilidad de que cancele.
- booking_changes ($-0.09$): Modificar la reserva reduce el riesgo de cancelación.

In [ ]:
df_clean = df_train.copy()
df_clean['babies'] = np.where(
        (df_clean['babies'] > 3) & (df_clean['adults'] <= 2), 
        1, 
        df_clean['babies']
    )

In [ ]:
# Seleccionar solo las variables numéricas relevantes
num_cols = [
    'is_canceled', 'lead_time', 'adr', 'required_car_parking_spaces', 
    'total_of_special_requests', 'booking_changes', 'is_repeated_guest', 
    'previous_cancellations', 'previous_bookings_not_canceled', 
    'stays_in_week_nights', 'stays_in_weekend_nights', 'adults', 'children', 'babies'
]

corr_matrix = df_clean[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Matriz de Correlación de Pearson (Variables Numéricas)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.2. Relacionar cada predictor con el target
¿Cómo afectan distintas variables a la tasa de cancelación?

**Hipótesis 1**: A mayor tiempo de anticipación (`lead_time`), mayor es la probabilidad de cancelación
- Pregunta: ¿Las reservas realizadas con muchos días de anticipación presentan un riesgo significativamente mayor de ser canceladas?
- Evidencia Cuantitativa (Estadísticos de Tukey):
  - Reservas No Canceladas ($0$): Mediana de 44 días de anticipación ($Q_1 = 8$, $Q_3 = 121$).
  - Reservas Canceladas ($1$): Mediana de 90 días de anticipación ($Q_1 = 37$, $Q_3 = 173$).
  - Promedio de días: $76.1$ días para no canceladas vs. $114.2$ días para canceladas.
- Conclusión: Hipótesis Confirmada. Las reservas que terminan en cancelación se realizan, en promedio, con más del doble de anticipación que las que se efectivizan. lead_time será uno de los predictores numéricos más potentes para los modelos de árbol y lineales.

**Hipótesis 2**: El tipo de depósito (`deposit_type`) determina fuertemente el comportamiento de cancelación
- Pregunta: ¿El hecho de exigir un depósito no reembolsable desincentiva la cancelación?
- Evidencia Cuantitativa (Tasa de Cancelación por Categoría):
  - No Deposit: Tasa de cancelación del 26.6% (73,184 reservas).
  - Refundable: Tasa de cancelación del 15.8% (95 reservas).
  - Non Refund: Tasa de cancelación extrema del 94.6% (1,007 reservas).
- Conclusión: Sorprendentemente, la categoría Non Refund presenta una tasa de cancelación de casi el 95% en los datos históricos. Esto suele indicar un patrón operativo del hotel donde las reservas de agencias de grupos o tarifas promocionales fijas son marcadas como Non Refund y masivamente canceladas por la propia agencia ante cambios de itinerario.

**Hipótesis 3**: La tasa de cancelación varía según el canal de venta (`market_segment`)
- Pregunta: ¿Las reservas digitales (OTAs) cancelan más que las corporativas o directas?
- Evidencia Cuantitativa (Tasa de Cancelación por Segmento):
  - Online TA (Booking, Expedia, etc.): 35.7% de cancelación (segmento mayoritario con 43,288 reservas).
  - Groups: 27.5% de cancelación.
  - Offline TA/TO: 14.7% de cancelación.
  - Direct: 14.5% de cancelación.
  - Corporate: 11.4% de cancelación.
- Conclusión: Hipótesis Confirmada. El canal online (Online TA) duplica la tasa de cancelación de los canales directos o corporativos. La flexibilidad y facilidad de cancelación sin costo en plataformas digitales es un motor directo del riesgo de cancelación.

**Hipótesis 4**: El compromiso del cliente se evidencia en las modificaciones de la reserva (`booking_changes`)
- Pregunta: ¿Un cliente que modifica su reserva tiene menor probabilidad de cancelarla?
- Evidencia Cuantitativa:
  - Sin Modificaciones (booking_changes = 0): Tasa de cancelación del 30.2% (60,682 reservas).
  - Con al menos 1 Modificación (booking_changes >= 1): Tasa de cancelación del 15.9% (13,604 reservas).
- Conclusión: Hipótesis Confirmada. Realizar cambios en la reserva (modificar fechas, tipo de habitación, peticiones) reduce la probabilidad de cancelación a la mitad. Esto demuestra una clara intención de consumo por parte del cliente.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# A. dep_cancel: Tasa de cancelación según Tipo de Depósito
dep_cancel = df_train.groupby('deposit_type')['is_canceled'].agg(['count', 'mean']).reset_index()
dep_cancel['mean'] = dep_cancel['mean'] * 100

# B. ms_cancel y ms_filtered: Tasa de cancelación según Segmento de Mercado
ms_cancel = df_train.groupby('market_segment')['is_canceled'].agg(['count', 'mean']).reset_index()
ms_cancel['mean'] = ms_cancel['mean'] * 100
ms_filtered = ms_cancel[ms_cancel['count'] > 10].sort_values(by='mean', ascending=False)

# C. bc_cancel: Tasa de cancelación según Modificaciones en la reserva (booking_changes > 0)
df_train['has_booking_changes'] = df_train['booking_changes'] > 0
bc_cancel = df_train.groupby('has_booking_changes')['is_canceled'].agg(['count', 'mean']).reset_index()
bc_cancel['mean'] = bc_cancel['mean'] * 100

# Panel 1: Lead Time vs is_canceled (Boxplot)
sns.boxplot(
    data=df_train, 
    x='is_canceled', 
    y='lead_time', 
    ax=axes[0, 0], 
    palette=['#2b5c8f', '#d95f02'], 
    hue='is_canceled'
)
axes[0, 0].set_title('Hipótesis 1: Días de Anticipación (lead_time) vs Cancelación', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Estado de Reserva', fontsize=10)
axes[0, 0].set_ylabel('Lead Time (Días)', fontsize=10)
axes[0, 0].set_xticks([0, 1])
axes[0, 0].set_xticklabels(['No Cancelada (0)', 'Cancelada (1)'])

# Panel 2: Deposit Type vs Tasa de Cancelación (%) + Conteo n
sns.barplot(data=dep_cancel, x='deposit_type', y='mean', ax=axes[0, 1], palette='Reds_r', hue='deposit_type')
axes[0, 1].set_title('Hipótesis 2: Tasa de Cancelación (%) según Tipo de Depósito', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Tipo de Depósito', fontsize=10)
axes[0, 1].set_ylabel('% de Cancelación', fontsize=10)

for p, count in zip(axes[0, 1].patches, dep_cancel['count']):
    h = p.get_height()
    if h > 0:
        y_pos = h / 2 if h > 30 else h + 5
        color_txt = 'white' if h > 30 else 'black'
        axes[0, 1].annotate(f'{h:.1f}%\n(n={count:,})', (p.get_x() + p.get_width() / 2., y_pos),
                             ha='center', va='center', color=color_txt, fontweight='bold', fontsize=10)

# Panel 3: Market Segment vs Tasa de Cancelación (%) + Conteo n
sns.barplot(
    data=ms_filtered, 
    x='mean', 
    y='market_segment', 
    ax=axes[1, 0], 
    palette='viridis', 
    hue='market_segment'
)
axes[1, 0].set_title('Hipótesis 3: Tasa de Cancelación (%) por Segmento de Mercado', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('% de Cancelación', fontsize=10)
axes[1, 0].set_ylabel('Segmento de Mercado', fontsize=10)

for p, count in zip(axes[1, 0].patches, ms_filtered['count']):
    w = p.get_width()
    if w > 0:
        x_pos = w / 2 if w > 20 else w + 4
        color_txt = 'white' if w > 20 else 'black'
        axes[1, 0].annotate(f'{w:.1f}% (n={count:,})', (x_pos, p.get_y() + p.get_height() / 2.),
                             ha='center', va='center', color=color_txt, fontweight='bold', fontsize=9)

# Panel 4: Modificaciones en Reserva vs Tasa de Cancelación (%)
sns.barplot(
    data=bc_cancel, 
    x='has_booking_changes', 
    y='mean', 
    ax=axes[1, 1], 
    palette=['#d95f02', '#2b5c8f'], 
    hue='has_booking_changes'
)
axes[1, 1].set_title('Hipótesis 4: Impacto de Modificaciones (booking_changes > 0)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('¿Se realizaron cambios en la reserva?', fontsize=10)
axes[1, 1].set_ylabel('% de Cancelación', fontsize=10)
axes[1, 1].set_xticks([0, 1])
axes[1, 1].set_xticklabels(['Sin Cambios (0)', 'Con Cambios (>=1)'])

for p in axes[1, 1].patches:
    h = p.get_height()
    if h > 0:
        axes[1, 1].annotate(
            f'{h:.1f}%', 
            (p.get_x() + p.get_width() / 2., h / 2),
            ha='center', va='center', color='white', fontweight='bold', fontsize=11
        )

plt.tight_layout()
plt.show()

In [ ]:

# 1. Reservas sin huéspedes y is_canceled
df_train['sin_huespedes'] = (df_train['adults'] == 0) & (df_train['children'] == 0) & (df_train['babies'] == 0)
rel_3 = df_train.groupby('sin_huespedes')['is_canceled'].agg(['count', 'sum', 'mean']).reset_index()
rel_3['pct_cancelado'] = rel_3['mean'] * 100

# 2. int(bool(days_in_waiting_list)) vs is_canceled
df_train['is_placed_on_waiting_list'] = (df_train['days_in_waiting_list'] > 0).astype(int)
rel_5 = df_train.groupby('is_placed_on_waiting_list')['is_canceled'].agg(['count', 'sum', 'mean']).reset_index()
rel_5['pct_cancelado'] = rel_5['mean'] * 100


print("\n1. Sin Huéspedes:\n", rel_3)
print("\n2. Lista de Espera Binarizada:\n", rel_5)

#"Graficos"

rel_3['huespedes_label'] = rel_3['sin_huespedes'].map({False: 'Con Huéspedes', True: 'Sin Huéspedes'})
rel_5['waiting_list_label'] = rel_5['is_placed_on_waiting_list'].map({0: 'No (0 días)', 1: 'Sí (> 0 días)'})

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Reservas sin huéspedes vs is_canceled
sns.barplot(data=rel_3, x='sin_huespedes', y='pct_cancelado', ax=axes[0], palette='Greens_r', hue='huespedes_label')
axes[0].set_title('1. Tasa de Cancelación en Reservas Sin Huéspedes', fontsize=11, fontweight='bold')
axes[0].set_xlabel('¿Reserva Sin Huéspedes?')
axes[0].set_ylabel('% Cancelación')
axes[0].set_xticklabels(['Con Huéspedes', 'Sin Huéspedes'])

# 2. Lista de Espera Binarizada vs is_canceled
sns.barplot(data=rel_5, x='is_placed_on_waiting_list', y='pct_cancelado', ax=axes[1], palette='Purples_r', hue='waiting_list_label' )
axes[1].set_title('2. Impacto de Lista de Espera en Cancelaciones', fontsize=11, fontweight='bold')
axes[1].set_xlabel('¿Estuvo en Lista de Espera?')
axes[1].set_ylabel('% Cancelación')
axes[1].set_xticklabels(['No (0 días)', 'Sí (> 0 días)'])

plt.tight_layout()
plt.show()

In [ ]:
# Calcular el % de cancelación por año
year_cancel = df_train.groupby('arrival_date_year')['is_canceled'].mean() * 100

# Configurar y crear el gráfico
plt.figure(figsize=(8, 4))
sns.barplot(x=year_cancel.index, y=year_cancel.values, palette="viridis")

plt.title('Porcentaje de Cancelación por Año')
plt.ylabel('% de Cancelación')
plt.xlabel('Año')
plt.tight_layout()
plt.show()

In [ ]:
# Orden cronológico de los meses
meses_orden = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']

# Calcular el % de cancelación por mes y reindexar
month_cancel = df_train.groupby('arrival_date_month')['is_canceled'].mean() * 100
month_cancel = month_cancel.reindex(meses_orden)

# Configurar y crear el gráfico
plt.figure(figsize=(12, 5))
sns.barplot(x=month_cancel.index, y=month_cancel.values, palette="magma")

plt.title('Porcentaje de Cancelación por Mes')
plt.ylabel('% de Cancelación')
plt.xlabel('Mes')
plt.xticks(rotation=45) # Rotamos las etiquetas para mejor lectura
plt.tight_layout()
plt.show()

In [ ]:
# Calcular el % de cancelación por día del mes
day_cancel = df_train.groupby('arrival_date_day_of_month')['is_canceled'].mean() * 100

# Configurar y crear el gráfico
plt.figure(figsize=(14, 5))
sns.barplot(x=day_cancel.index, y=day_cancel.values, palette="coolwarm")

plt.title('Porcentaje de Cancelación por Día del Mes')
plt.ylabel('% de Cancelación')
plt.xlabel('Día del Mes')
plt.tight_layout()
plt.show()

# 4. Análisis multivarido

**Interacción 1**: `lead_time` $\times$ `market_segment` vs `is_canceled`
Al cruzar los días de anticipación con el canal de venta y el estado de cancelación, descubrimos que el efecto del lead_time no es uniforme entre segmentos en cuanto a 'valores':
- Segmento Groups (Grupos): Exhibe la mayor anticipación de todo el dataset. Las reservas de grupo que terminan cancelándose tienen una mediana de 164 días de anticipación, frente a 112 días en las que se efectivizan.
- Segmento Online TA (Agencias Online): El riesgo de cancelación se dispara sustancialmente cuando el lead_time supera los 89 días (mediana de canceladas), mientras que las reservas no canceladas tienen una mediana de solo 47 días.
- Segmento Corporate (Empresas): Mantiene una anticipación sumamente baja (mediana de 5 días para no canceladas y 12 días para canceladas), conservando tasas de cancelación reducidas en cualquier rango de tiempo.

Pero si se observa un patrón constante en todos los segmentos del mercado, por que podemos concluir:
- Efecto Aditivo / Independiente: Esto nos confirma que lead_time tiene un efecto directo por sí solo (a menor anticipación, menor probabilidad de cancelación independientemente del canal).
- Desplazamiento de la Escala (Offset por Segmento): Si bien el patrón intrínseco se repite (los que cancelan tienen un lead_time más alto), la escala de tiempo absoluta de cada segmento es radicalmente distinta.
  - Una reserva de 90 días de anticipación en el canal Direct representa un riesgo alto (está muy por encima de la mediana de cancelados de 45 días).
  - Esa misma reserva de 90 días de anticipación en el canal Groups representa un riesgo bajo (está por debajo de la mediana de los que efectivizan su estancia, que es de 112 días).

**Interacción 2**: `hotel` $\times$ `market_segment` vs `is_canceled `
Al evaluar el comportamiento cruzado entre el tipo de establecimiento (City Hotel vs Resort Hotel) y el canal de comercialización, se observan variaciones en la tasa de cancelación:
- Efecto en Segmento Groups: En City Hotel, la tasa de cancelación para grupos alcanza un elevadísimo 34.6%, mientras que en Resort Hotel desciende drásticamente al 19.6%.
- Efecto en Segmento Online TA: En ambos hoteles representa el canal de mayor riesgo, registrando un 36.5% de cancelación en City Hotel y un 34.0% en Resort Hotel.
- Efecto en Canales Directos (Direct / Corporate): Ambos tipos de hotel logran mantener tasas de cancelación bajas (entre el 10% y el 16%).

In [ ]:
plt.figure(figsize=(16, 10))

# Panel 1: Lead Time vs Segmento de Mercado por Estado de Cancelación
plt.subplot(2, 1, 1)
sns.boxplot(
    data=df_train[df_train['market_segment'].isin(['Online TA', 'Offline TA/TO', 'Direct', 'Groups', 'Corporate'])],
    x='market_segment',
    y='lead_time',
    hue='is_canceled',
    palette=['#2b5c8f', '#d95f02']
)
plt.title('Interacción 1: Lead Time vs Segmento de Mercado por Estado de Cancelación', fontsize=12, fontweight='bold')
plt.xlabel('Segmento de Mercado', fontsize=10)
plt.ylabel('Lead Time (Días)', fontsize=10)
plt.legend(title='is_canceled')

# Panel 2: Tasa de Cancelación (%) por Segmento de Mercado y Tipo de Hotel
multivar_adr = df_train.groupby(['hotel', 'market_segment'])['is_canceled'].agg(['count', 'mean']).reset_index()
multivar_adr = multivar_adr[multivar_adr['count'] > 100]
multivar_adr['pct'] = multivar_adr['mean'] * 100

plt.subplot(2, 1, 2)
sns.barplot(
    data=multivar_adr,
    x='market_segment',
    y='pct',
    hue='hotel',
    palette=['#2b5c8f', '#41b6c4']
)
plt.title('Interacción 2: Tasa de Cancelación (%) por Segmento de Mercado y Tipo de Hotel', fontsize=12, fontweight='bold')
plt.xlabel('Segmento de Mercado', fontsize=10)
plt.ylabel('% de Cancelación', fontsize=10)
plt.legend(title='Tipo de Hotel')

for p in plt.gca().patches:
    h = p.get_height()
    if not np.isnan(h) and h > 0:
        plt.gca().annotate(f'{h:.1f}%', (p.get_x() + p.get_width() / 2., h / 2),
                           ha='center', va='center', color='white', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:

# 1. adr == 0 con customer_type y is_canceled
adr_zero = df_train[df_train['adr'] == 0]
rel_1 = adr_zero.groupby('customer_type')['is_canceled'].agg(['count', 'sum', 'mean']).reset_index()
rel_1['pct_cancelado'] = rel_1['mean'] * 100

# 2. adr (general) con customer_type y is_canceled
rel_2 = df_train.groupby('customer_type').agg(
    total_reservas=('is_canceled', 'count'),
    tasa_cancelacion_pct=('is_canceled', lambda x: x.mean() * 100),
    adr_promedio=('adr', 'mean'),
    adr_mediana=('adr', 'median')
).reset_index()

print("1. ADR == 0:\n", rel_1)
print("\n2. ADR General:\n", rel_2)

#Gráficos

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. ADR == 0 vs Tasa de Cancelación por customer_type
sns.barplot(data=rel_1, x='customer_type', y='pct_cancelado', ax=axes[0], palette='Blues_r', hue='customer_type')
axes[0].set_title('1. Tasa de Cancelación (%) en Reservas con ADR = 0', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Tipo de Cliente')
axes[0].set_ylabel('% Cancelación')

# 2. Tasa de Cancelación General por customer_type
sns.barplot(data=rel_2, x='customer_type', y='tasa_cancelacion_pct', ax=axes[1], palette='Oranges_r', hue='customer_type')
for p, count, adr in zip(axes[1].patches, rel_2['total_reservas'], rel_2['adr_promedio']):
    height = p.get_height()
    
    # Texto de 3 líneas con el formato exacto
    texto = f"{height:.1f}%\n(n={count:,})\nADR: {adr:.1f}€"
    
    x = p.get_x() + p.get_width() / 2.0
    y = height / 2.0
    
    axes[1].annotate(
        texto,
        (x, y),
        ha='center', va='center',
        color='white' if height > 12 else 'black',
        fontweight='bold',
        fontsize=10
    )
axes[1].set_title('2. Tasa de Cancelación General (%) por Tipo de Cliente', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Tipo de Cliente')
axes[1].set_ylabel('% Cancelación')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Agrupamiento rel_4 tal como lo planteaste
df_train['sin_huespedes'] = (df_train['adults'] == 0) & (df_train['children'] == 0) & (df_train['babies'] == 0)

# 2. Reservas sin huéspedes por ADR y is_canceled
rel_4 = df_train.groupby(['sin_huespedes', 'is_canceled']).agg(
    cantidad=('adr', 'count'),
    adr_promedio=('adr', 'mean'),
    adr_mediana=('adr', 'median')
).reset_index()


print("\n2. Sin Huéspedes por ADR:\n", rel_4)

# 3. Mapeo de etiquetas legibles para la visualización
rel_4['huespedes_label'] = rel_4['sin_huespedes'].map({False: 'Con Huéspedes', True: 'Sin Huéspedes'})
rel_4['canceled_label'] = rel_4['is_canceled'].map({0: 'No Canceló (0)', 1: 'Canceló (1)'})

# 4. Creación de la figura con 2 subplots horizontales
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# -------------------------------------------------------------------------
# GRAFICO 1: Cantidad Total de Reservas (Eje Y = cantidad)
# -------------------------------------------------------------------------
sns.barplot(
    data=rel_4,
    x='huespedes_label',
    y='cantidad',
    hue='canceled_label',
    ax=axes[0],
    palette=['#2b5c8f', '#d95f02']
)
axes[0].set_title('1. Cantidad Total de Reservas por Estado y Huéspedes', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Presencia de Huéspedes', fontsize=10)
axes[0].set_ylabel('Cantidad de Reservas (n)', fontsize=10)
axes[0].legend(title='Estado de Reserva')

# Anotaciones con la cantidad de reservas sobre cada barra
for p in axes[0].patches:
    h = p.get_height()
    if not np.isnan(h) and h > 0:
        # Si la barra es muy pequeña (Sin Huéspedes), colocamos el texto arriba para garantizar legibilidad
        y_pos = h / 2.0 if h > 5000 else h + 1000
        color_txt = 'white' if h > 5000 else 'black'
        axes[0].annotate(
            f'{int(h):,}',
            (p.get_x() + p.get_width() / 2.0, y_pos),
            ha='center', va='center',
            color=color_txt, fontweight='bold', fontsize=10
        )

# -------------------------------------------------------------------------
# GRAFICO 2: ADR Promedio (Eje Y = adr_promedio)
# -------------------------------------------------------------------------
sns.barplot(
    data=rel_4,
    x='huespedes_label',
    y='adr_promedio',
    hue='canceled_label',
    ax=axes[1],
    palette=['#2b5c8f', '#d95f02']
)
axes[1].set_title('2. ADR Promedio (€) por Estado y Huéspedes', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Presencia de Huéspedes', fontsize=10)
axes[1].set_ylabel('ADR Promedio (EUR)', fontsize=10)
axes[1].legend(title='Estado de Reserva')

# Anotaciones con el ADR promedio (€) sobre cada barra
for p in axes[1].patches:
    h = p.get_height()
    if not np.isnan(h):
        y_pos = h / 2.0 if h > 20 else h + 3.0
        color_txt = 'white' if h > 20 else 'black'
        axes[1].annotate(
            f'{h:.2f}€',
            (p.get_x() + p.get_width() / 2.0, y_pos),
            ha='center', va='center',
            color=color_txt, fontweight='bold', fontsize=10
        )

plt.tight_layout()
plt.show()

# 5. Detectar transformaciones necesarias

A partir del análisis de distribuciones y sesgos cuantitativos sobre df_train, identificamos las siguientes transformaciones previas a ingresar a Scikit-Learn:
- Escalado Robusto / Logarítmico:
  - lead_time: Asimetría positiva severa (right-skewed, asimetría $> 1.2$). Aplicar RobustScaler() o transformación $\log(1 + x)$ para algoritmos sensibles como Logistic Regression o KNN.
  - adr: Presenta dispersión con atípicos extremos. Requiere acotamiento (clipping a $[0, 1000]$ EUR) y posterior RobustScaler().
- Codificación Categórica:
  - Baja Cardinalidad (hotel, customer_type, deposit_type, distribution_channel): OneHotEncoder(drop='first', handle_unknown='ignore').
  - Media/Alta Cardinalidad (market_segment, reserved_room_type, assigned_room_type): OneHotEncoder().
  - Alta Cardinalidad (country): Reagrupar manteniendo los Top 10-15 países con mayor volumen y asignar el resto a 'Other' antes de aplicar One-Hot Encoding.
- Codificación Categórica Ordinal:
  - meal: Mapeo ordinal (SC/Undefined $\rightarrow 0$, BB $\rightarrow 1$, HB $\rightarrow 2$, FB $\rightarrow 3$).
  - arrival_date_month: Mapeo cíclico sinusoidal ($\sin/\cos$) o numérico continuo ($1$ a $12$).


# 6. Analizar temporalidad

- Secuencia de Registro: El evento real de reserva ocurre en la fecha sintética $\text{booking\_date} = \text{arrival\_date} - \text{lead\_time}$.
- Estabilidad del Target: La tasa mensual de cancelación se mantiene estable en la serie histórica a partir de 2016 (alrededor del $27\% - 30\%$).
- Patrón de Ocupación Futura: Las cancelaciones aumentan conforme el tiempo de anticipación (lead_time) se extiende, lo que ratifica la necesidad de la partición temporal estricta para evaluar la generalización sin alterar la línea de tiempo del negocio.


# 7. Evaluar fuga de información (Data Leakage)

Identificamos variables que ocurren posteriormente al evento de predicción y deben ser eliminadas inmediatamente del espacio de características ($X$):
- `reservation_status`: Indica directamente el resultado administrativo (Check-Out, Canceled, No-Show). Su presencia otorgaría un Accuracy perfecto pero falso (Leakage directo).
- `reservation_status_date`: Registra el día exacto en que se efectivizó la cancelación o salida. Es un dato posterior al momento de la reserva.
- `assigned_room_type`: La asignación de la habitación es posterior al momento de la reserva, correponde al día en que se hace el check-in.
- `booking_changes`: Los cambios en la reserva se realizan luego de haberla hecho. 
- `days_in_waiting_list` (se transforma a is_placed_on_waiting_list): Al momento de la reserva no se puede conocer la cantidad de días que estará en lista de espera, suponiendo que el hotel es quien pone en lista de espera una determinada reserva por no tener lugar en ese momento, pero si podemos saber al momento de hacer la reserva si va a pasar a la lista de espera o es confirmada directamente. Se tiene como fundamento para este supuesto que en niguna observación los días en `days_in_waiting_list` han sido más que el `lead_time` (días transcurridos entre la reserva y la llegada), por lo que se concluyó que el `lead_time` incluye `days_in_waiting_list`.

# 8. Identificación y clasificación de variables candidatas para el modelo
Variables de Alta Prioridad (Alta señal predictiva + Disponibilidad real en el momento de la reserva):
- `lead_time`: Fuerte correlación positiva ($+0.19$) y clara separación en medianas (44 días en no canceladas vs. 90 días en canceladas).
- `required_car_parking_spaces`: Alta correlación negativa ($-0.18$). Solicitar parking reduce drásticamente el riesgo de cancelación.
- `total_of_special_requests`: Correlación negativa ($-0.12$). Refleja compromiso directo del huésped.
- `market_segment` / `distribution_channel`: Las reservas vía Online TA duplican el riesgo de cancelación respecto a canales directos/corporativos.
- `deposit_type`: Distingue reservas garantizadas vs. sin depósito.
- `adr`: Factor de sensibilidad al precio ($+0.12$ de correlación).
- `booking_date`, `arrival_month_num`, `arrival_date_year`


Variables de Prioridad Media (Asociación moderada o señal complementaria):
- `hotel`: Define el perfil del huésped (Urbano vs. Vacacional).
- `customer_type`: Distingue reservas individuales (Transient) de corporativas/contratos.
- `previous_cancellations` y `previous_bookings_not_canceled`: Histórico del cliente (va una sola)
- `stays_in_week_nights` y `stays_in_weekend_nights`: Duración de la estadía.
- `adults`, `children`, `babies`: Composición del grupo.
- `country`: País de origen (tras agrupamiento en Top $N$ + 'Other').

Variables de Baja Prioridad (Poca variabilidad, nula señal predictiva o alto riesgo de sobreajuste):
- `arrival_date_day_of_month` y `arrival_date_week_number`: Correlación cercana a cero (ruido blanco). No van 
- `agent` y `company`: Identificadores con cardinalidad masiva ($> 300$ categorías únicas) y alto porcentaje de faltantes.

# 9. Conclusiones del EDA y recomendaciones de modelado

### 9.1. Recomendaciones de Preprocesamiento:
1. Observación Inicial: 
  - Conservar reservas con $0$ huéspedes totales (128 observaciones - 115 no canceladas) - Se considera que al momento de hacer la reserva el huesped no sabe con certeza cuantas personas se vana hospedar.
  - Analizar si quitar el ruido que introducen los datos anteriores al 2015 o no.

2. Tratamiento de Nulos:
  - children $\rightarrow$ Imputar con 0.
  - country $\rightarrow$ Imputar con 'Unknown' y reducir categorías a `is_portugal`.
  - agent / company $\rightarrow$ Imputar con 0 (tratar 0 como "Sin Agente/Sin Empresa").

3. Tratamiento de Outliers: 
  - Regla personalizada sobre ADR:
    1. Si es negativo (< 0), se imputa en 0.0.
    2. Si supera 400.0 EUR, se imputa con la mediana de las tarifas normales.
    3. De lo contrario, conserva su valor original.
  - Regla condicional para babies:
    $$\text{babies\_corregido} = \begin{cases}  1 & \text{si } \text{babies} > 3 \text{ y } \text{adults} \le 2 \\  \text{babies} & \text{en cualquier otro caso} \end{cases}$$
  - children: se consideran a los menores de edad (hasta 17 años) por lo que no se toma ninguna acción al respecto, pueden hacer una reserva sin adultos. 
  - stays_in_week_nights y stays_in_weekend_nights: se han analizado en conjunto y no presentan incongruencian en las observaciones, por lo que no se toma ninguna acción al respecto.

4. Feature Engineering Sugerido:
  - total_nights = stays_in_week_nights + stays_in_weekend_nights, usamos la suma porque la correlación de pearson entre ellas es 0.54 y eso nos indica que hay información entre ellas que se esta duplicando. 
  - total_guests = adults + children + babies, optamos por dejar las variables por separado ya que la correlación con respecto a `is_canceled` de adults y children (0.07) es diferente que con babies (-0.02). Si resumiéramos las tres variables en una sola la relación con `is_canceled` se suavizaría
  - is_placed_on_waiting_list = int(bool(days_in_waiting_list))
  - previous_cancellations y previous_bookings_not_canceled son complementarias, elegimos usar previous_cancellations porque tiene mejor correlación de pearson con el resto de las variables. 

### 9.2. Arquitectura del Pipeline de Scikit-Learn:
- ColumnTransformer:
  - Rama Numérica: SimpleImputer(strategy='median') $\rightarrow$ RobustScaler()
  - Rama Categórica Nominal: SimpleImputer(strategy='most_frequent') $\rightarrow$ OneHotEncoder(drop='first', handle_unknown='ignore')

### 9.3. Estrategia de Evaluación:
- Evitar la métrica Accuracy por el desbalance moderado ($2.63 : 1$).
- Priorizar ROC-AUC y F1-Score sobre la clase $1$ (Cancelada).
- Configurar class_weight='balanced' en los algoritmos candidatos.


# Notas auxiliares

In [ ]:

def apply_initial_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aplica las reglas de limpieza lógica, imputación de faltantes y corrección de anómalos.
    """
    df_clean = df.copy()
    
    # 1. Imputación de valores faltantes
    df_clean['children'] = df_clean['children'].fillna(0)
    df_clean['country'] = df_clean['country'].fillna('Unknown')
    df_clean['agent'] = df_clean['agent'].fillna(0)
    df_clean['company'] = df_clean['company'].fillna(0)
    
    # 2. Regla personalizada para ADR (Tarifa diaria promedio)
    # Si ADR < 0 pasa a 0.0; si ADR > 400.0 se imputa por la mediana representativa
    # valid_adr_median = df_clean.loc[(df_clean['adr'] >= 0) & (df_clean['adr'] <= 400), 'adr'].median()
    # df_clean['adr'] = np.where(
    #     df_clean['adr'] < 0, 
    #     0.0, 
    #     np.where(df_clean['adr'] > 400, valid_adr_median, df_clean['adr'])
    # )
    # Imputar solo la mediana
    
    # 3. Regla condicional para la variable 'babies'
    df_clean['babies'] = np.where(
        (df_clean['babies'] > 3) & (df_clean['adults'] <= 2), 
        1, 
        df_clean['babies']
    )
    
    # 4. Eliminación de columnas con Data Leakage
    cols_to_drop = ['reservation_status', 'reservation_status_date', 'assigned_room_type', 'booking_changes', 'days_in_waiting_list']
    df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns], inplace=True)
    
    return df_clean;

apply_initial_cleaning(df_train)